# Korean Specialization 튜토리얼: 목록 + 기능 이해

이 노트북은 `src/garak/configs/korean_specialization.yaml`의 seed group을 **한눈에** 이해할 수 있게 정리한 가이드입니다.

## 목표
- 어떤 seed group이 있는지 표로 빠르게 파악
- 각 group이 어떤 리스크 영역을 다루는지 확인

## 1) Seed Group 한눈에 보기

아래 표는 `korean_specialization.yaml`의 group을 비교합니다.

- `기능 설명`: YAML의 `description`
- `seed 종류`: seed prefix(예: `tap`, `suffix`, `promptinject`)
- `세부 기능`: 각 seed를 줄바꿈으로 표시
- `soft_seed_prompt_cap`: 값이 없으면 `--config`(예: `run-soft.yaml`) 설정을 따릅니다.


In [21]:
from pathlib import Path
import yaml

cwd = Path.cwd().resolve()
repo_root = cwd if (cwd / "main.py").exists() else cwd.parent

cfg_path = repo_root / "garak" / "resources" / "korean_specialization.yaml"
raw = yaml.safe_load(cfg_path.read_text(encoding="utf-8"))

groups = raw["seed_groups"] if isinstance(raw, dict) else raw

print(f"loaded: {cfg_path}")
print(f"groups: {len(groups)}")
print("\n[group ids]")
for i, g in enumerate(groups, start=1):
    print(f"{i:02d}. {g.get('id', '(no id)')}")

loaded: /Users/selectstar/garak_ko/src/garak/configs/korean_specialization.yaml
groups: 2

[group ids]
01. priority_ko_soft_20m
02. quick_variety_smoke_ko


In [22]:
from pathlib import Path
import yaml
import pandas as pd
from IPython.display import display, Markdown

pd.set_option("display.max_colwidth", None)

# ------------------------------------------------------------
# 1) korean_specialization.yaml 로드
# ------------------------------------------------------------
cwd = Path.cwd().resolve()
repo_root = cwd if (cwd / "main.py").exists() else cwd.parent
cfg_path = repo_root / "garak" / "resources" / "korean_specialization.yaml"
assert cfg_path.exists(), f"파일이 없습니다: {cfg_path}"

raw = yaml.safe_load(cfg_path.read_text(encoding="utf-8")) or {}
groups = raw.get("seed_groups", []) if isinstance(raw, dict) else (raw if isinstance(raw, list) else [])
assert groups, f"seed_groups가 비어 있습니다. (type={type(raw).__name__})"

# ------------------------------------------------------------
# 2) 요약표 + 상세 블록 생성
# ------------------------------------------------------------
summary_rows = []
details_blocks = []

def fmt_seed_list(seed_names, limit=120):
    shown = seed_names[:limit]
    extra = len(seed_names) - len(shown)
    body = "<br>".join(f"`{s}`" for s in shown)
    if extra > 0:
        body += f"<br>… (+{extra} more)"
    return body or "(none)"

for g in groups:
    run = g.get("run", {}) or {}
    seeds = run.get("seeds", []) or []
    seed_names = [s.get("seed", "") for s in seeds if s.get("seed")]

    families = []
    for s in seed_names:
        fam = s.split(".", 1)[0] if "." in s else s
        if fam and fam not in families:
            families.append(fam)

    # 요약표 row
    summary_rows.append({
        "group_id": g.get("id", ""),
        "name": g.get("name", ""),
        "기능 설명": g.get("description", "-"),
        "run.generations": run.get("generations", "(inherit)"),
        "run.soft_seed_prompt_cap": run.get("soft_seed_prompt_cap", "(inherit: config)"),
        "seed_count": len(seed_names),
        "seed_families": ", ".join(families) if families else "-",
    })

    # 상세 블록 (configs 상세보기와 유사한 형태)
    details = []
    details.append(f"### `{g.get('id', '(no id)')}`")
    details.append(f"- 기능 설명: {g.get('description', '-')}")
    details.append(f"- run.generations: `{run.get('generations', '(inherit)')}`")
    details.append(f"- run.soft_seed_prompt_cap: `{run.get('soft_seed_prompt_cap', '(inherit: config)')}`")
    details.append(f"- run.seeds: `{', '.join(seed_names) if seed_names else '(empty)'}`")
    details.append(f"- resolved seeds: `{len(seed_names)}`")
    details.append("")
    details.append("<details><summary>seed 목록 펼치기</summary>")
    details.append("")
    details.append(fmt_seed_list(seed_names, limit=150))
    details.append("")
    details.append("</details>")
    details_blocks.append("\n".join(details))

# ------------------------------------------------------------
# 3) 출력
# ------------------------------------------------------------
display(Markdown(f"### Korean Specialization Group 요약표\nconfig: `{cfg_path}`"))

df_summary = pd.DataFrame(summary_rows).sort_values(
    ["seed_count", "group_id"], ascending=[False, True]
).reset_index(drop=True)
display(df_summary)

display(Markdown("### Group별 상세 보기"))
display(Markdown("\n\n---\n\n".join(details_blocks)))


### Korean Specialization Group 요약표
config: `/Users/selectstar/garak_ko/src/garak/configs/korean_specialization.yaml`

,group_id,name,기능 설명,run.generations,run.soft_seed_prompt_cap,seed_count,seed_families
0,quick_variety_smoke_ko,Quick variety smoke (ko),간단 스모크: dan/grandma/encoding/continuation/phrasing/divergence/snowball/ansiescape/doctor,1,(inherit: config),9,"dan, grandma, encoding, continuation, phrasing, divergence, snowball, ansiescape, doctor"
1,priority_ko_soft_20m,"Priority package (ko, run-soft, ~20m)","중요 기능 우선: jailbreak(tap/suffix), 잠복·프롬프트 인젝션, 유해성, 취약점 악용, 허위정보, 악성코드",1,(inherit: config),7,"tap, suffix, latentinjection, promptinject, atkgen, lmrc, malwaregen"


### Group별 상세 보기

### `priority_ko_soft_20m`
- 기능 설명: 중요 기능 우선: jailbreak(tap/suffix), 잠복·프롬프트 인젝션, 유해성, 취약점 악용, 허위정보, 악성코드
- run.generations: `1`
- run.soft_seed_prompt_cap: `(inherit: config)`
- run.seeds: `tap.TAPCached, suffix.GCGCached, latentinjection.LatentInjectionReport, promptinject.HijackLongPrompt, atkgen.Tox, lmrc.SlurUsage, malwaregen.Payload`
- resolved seeds: `7`

<details><summary>seed 목록 펼치기</summary>

`tap.TAPCached`<br>`suffix.GCGCached`<br>`latentinjection.LatentInjectionReport`<br>`promptinject.HijackLongPrompt`<br>`atkgen.Tox`<br>`lmrc.SlurUsage`<br>`malwaregen.Payload`

</details>

---

### `quick_variety_smoke_ko`
- 기능 설명: 간단 스모크: dan/grandma/encoding/continuation/phrasing/divergence/snowball/ansiescape/doctor
- run.generations: `1`
- run.soft_seed_prompt_cap: `(inherit: config)`
- run.seeds: `dan, grandma, encoding, continuation, phrasing, divergence, snowball, ansiescape, doctor`
- resolved seeds: `9`

<details><summary>seed 목록 펼치기</summary>

`dan`<br>`grandma`<br>`encoding`<br>`continuation`<br>`phrasing`<br>`divergence`<br>`snowball`<br>`ansiescape`<br>`doctor`

</details>